## Transformación y limpieza de datos

### Objetivo
Este notebook aplica las transformaciones necesarias sobre los datos extraídos para prepararlos para su carga en la base de datos analítica.

Las operaciones incluyen:
- **Tipado de columnas**: conversión de IDs numéricos a cadena (`str`) para tratarlos como variables categóricas.
- **Mapeo de territorios**: asignación del `id_territorio` desde la tabla dimensión a los DataFrames de hechos (constituidas y disueltas), y normalización de nombres (minúsculas, guiones bajos).
- **Normalización de texto**: estandarización de nombres de sectores, meses, razones de disolución y tipos de medida.
- **Eliminación de duplicados**: filtrado de filas "Mercantiles" que agregan información ya presente en los desgloses por tipo societario.
- **Abreviaturas**: conversión de tipos societarios a siglas (S.A., S.L., S.Com./S.C.).

### Metodología
1. **Carga** de los CSV desde `../files/data_raw/`.
2. **Transformaciones** aplicadas mediante funciones del módulo `src.transformacion` y mapeos manuales.
3. **Exportación** de los datasets procesados a `../files/data_processed/` para su consumo en la fase de carga.

In [256]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de transformacion
from src.transformacion import trans_str
from src.transformacion import trans_normal
# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)

Abrimos los ficheros

In [257]:
df_empr_const = pd.read_csv('../files/data_raw/empresas_constituidas.csv')
df_empr_dis = pd.read_csv('../files/data_raw/empresas_disueltas.csv')
df_ipc = pd.read_csv('../files/data_raw/ipc.csv')
df_sectores_ipc = pd.read_csv('../files/data_raw/sectores_ipc.csv')
df_territorio = pd.read_csv('../files/data_raw/territorio.csv')
df_tiempo = pd.read_csv('../files/data_raw/tiempo.csv')
df_tipo_medida = pd.read_csv('../files/data_raw/tipo_medida.csv')

Transformamos id's, año y mes a string ya que son falsas numericas (actuan como categoricas)

In [258]:
# Listas de columnas a transformar de cada DataFrame
lista_const = ['id_const', 'id_tiempo']
lista_dis = ['id_dis', 'id_tiempo']
lista_ipc = ['id_tiempo', 'id_territorio', 'id_sector', 'id_medida']
lista_sector_ipc = ['id_sector']
lista_territorio = ['id_territorio']
lista_tiempo = ['id_tiempo', 'anio', 'mes']
lista_tipo_medida = ['id_medida']

In [259]:
trans_str.int_a_str(df_empr_const, lista_const)
trans_str.int_a_str(df_empr_dis, lista_dis)
trans_str.int_a_str(df_ipc, lista_ipc)
trans_str.int_a_str(df_sectores_ipc, lista_sector_ipc)
trans_str.int_a_str(df_territorio, lista_territorio)
trans_str.int_a_str(df_tiempo, lista_tiempo)
trans_str.int_a_str(df_tipo_medida, lista_tipo_medida)

-> Columna 'id_const' convertida a str.
-> Columna 'id_tiempo' convertida a str.
-> Columna 'id_dis' convertida a str.
-> Columna 'id_tiempo' convertida a str.
-> Columna 'id_tiempo' convertida a str.
-> Columna 'id_territorio' convertida a str.
-> Columna 'id_sector' convertida a str.
-> Columna 'id_medida' convertida a str.
-> Columna 'id_sector' convertida a str.
-> Columna 'id_territorio' convertida a str.
-> Columna 'id_tiempo' convertida a str.
-> Columna 'anio' convertida a str.
-> Columna 'mes' convertida a str.
-> Columna 'id_medida' convertida a str.


,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [260]:
df_empr_const.info()

<class 'pandas.DataFrame'>
RangeIndex: 16720 entries, 0 to 16719
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_const           16720 non-null  str  
 1   territorio         16720 non-null  str  
 2   id_tiempo          16720 non-null  str  
 3   tipo               16720 non-null  str  
 4   numero_sociedades  16720 non-null  int64
 5   capital            16720 non-null  int64
dtypes: int64(2), str(4)
memory usage: 783.9 KB


In [261]:
df_empr_dis.info()

<class 'pandas.DataFrame'>
RangeIndex: 12540 entries, 0 to 12539
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_dis             12540 non-null  str  
 1   territorio         12540 non-null  str  
 2   id_tiempo          12540 non-null  str  
 3   razon              12540 non-null  str  
 4   numero_sociedades  12540 non-null  int64
dtypes: int64(1), str(4)
memory usage: 490.0 KB


In [262]:
df_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 311752 entries, 0 to 311751
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id_tiempo      311752 non-null  str    
 1   id_territorio  311752 non-null  str    
 2   id_sector      311752 non-null  str    
 3   id_medida      311752 non-null  str    
 4   valor_ipc      311752 non-null  float64
dtypes: float64(1), str(4)
memory usage: 11.9 MB


In [263]:
df_sectores_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_sector      14 non-null     str  
 1   nombre_sector  14 non-null     str  
dtypes: str(2)
memory usage: 356.0 bytes


In [264]:
df_territorio.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_territorio      20 non-null     str  
 1   nombre_territorio  20 non-null     str  
dtypes: str(2)
memory usage: 452.0 bytes


In [265]:
df_territorio.head()

,id_territorio,nombre_territorio
0,1,Nacional
1,2,Andalucía
2,3,Aragón
3,4,"Asturias, Principado de"
4,5,"Balears, Illes"


In [266]:
df_tiempo.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_tiempo   294 non-null    str  
 1   anio        294 non-null    str  
 2   mes         294 non-null    str  
 3   nombre_mes  294 non-null    str  
dtypes: str(4)
memory usage: 9.3 KB


In [267]:
df_tipo_medida.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_medida      4 non-null      str  
 1   nombre_medida  4 non-null      str  
dtypes: str(2)
memory usage: 196.0 bytes


- Cambiar los Nombres de las comunidades autónomas, sin acentos, con minúsculas y separación con guión bajo ('_')
- DataFrames: empresas_constituidas, empresas_disueltas y territorio

In [268]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
mapeo_territorios_nombre = {
    "Nacional": "nacional",
    "Andalucía": "andalucia",
    "Aragón": "aragon",
    "Asturias, Principado de": "principado_de_asturias",
    "Balears, Illes": "islas_baleares",
    "Canarias": "canarias",
    "Cantabria": "cantabria",
    "Castilla y León": "castilla_y_leon",
    "Castilla - La Mancha": "castilla_la_mancha",
    "Cataluña": "cataluna",
    "Comunitat Valenciana": "comunidad_valenciana",
    "Extremadura": "extremadura",
    "Galicia": "galicia",
    "Madrid, Comunidad de": "comunidad_de_madrid",
    "Murcia, Región de": "region_de_murcia",
    "Navarra, Comunidad Foral de": "comunidad_foral_de_navarra",
    "País Vasco": "pais_vasco",
    "Rioja, La": "la_rioja",
    "Ceuta": "ceuta",
    "Melilla": "melilla"
}

In [269]:
# Diccionario territorio -> id_territorio, usando el propio df_territorio
mapeo_territorios = dict(zip(df_territorio['nombre_territorio'], df_territorio['id_territorio']))

# Aplicar el mapeo
df_empr_const['id_territorio'] = df_empr_const['territorio'].map(mapeo_territorios)
df_empr_dis['id_territorio'] = df_empr_dis['territorio'].map(mapeo_territorios)

In [270]:
df_empr_const.sample(10)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital,id_territorio
2308,2309,Extremadura,201704,Mercantiles,125,1604000,12
3237,3238,"Navarra, Comunidad Foral de",201303,Mercantiles,47,847000,16
4615,4616,Aragón,200805,Sociedades anónimas,2,120000,3
12065,12066,"Rioja, La",201011,Sociedades de responsabilidad limitada,21,310000,18
9566,9567,Cantabria,201706,Sociedades de responsabilidad limitada,67,585000,7
9339,9340,Canarias,201801,Sociedades de responsabilidad limitada,342,10398000,6
15103,15104,Galicia,201405,S. Comanditarias y S. Colectivas,0,0,13
6333,6334,Comunitat Valenciana,201111,Sociedades anónimas,2,120000,11
6342,6343,Comunitat Valenciana,201102,Sociedades anónimas,6,1710000,11
14389,14390,Cataluña,201811,S. Comanditarias y S. Colectivas,0,0,10


In [271]:
df_empr_dis.sample(10)

,id_dis,territorio,id_tiempo,razon,numero_sociedades,id_territorio
7901,7902,"Rioja, La",200907,Por fusión,0,18
39,40,Andalucía,202301,Voluntaria,478,2
9227,9228,"Balears, Illes",200901,Otras,2,5
4062,4063,Melilla,201710,Voluntaria,1,20
9891,9892,Castilla y León,200809,Otras,1,8
12049,12050,"Rioja, La",201203,Otras,0,18
4209,4210,Andalucía,202311,Por fusión,22,2
6084,6085,Cataluña,201404,Por fusión,13,10
5696,5697,Castilla y León,200912,Por fusión,6,8
3209,3210,"Navarra, Comunidad Foral de",201507,Voluntaria,0,16


In [272]:
df_empr_dis.drop(columns=["territorio"],inplace=True)

In [273]:
df_empr_const.drop(columns=["territorio"],inplace=True)

In [274]:
#df_empr_const['territorio'] = df_empr_const['territorio'].replace(mapeo_territorios)
#df_empr_dis['territorio'] = df_empr_dis['territorio'].replace(mapeo_territorios)
df_territorio['nombre_territorio'] = df_territorio['nombre_territorio'].replace(mapeo_territorios_nombre)

In [275]:
df_territorio['nombre_territorio'].unique()

<StringArray>
[                  'nacional',                  'andalucia',
                     'aragon',     'principado_de_asturias',
             'islas_baleares',                   'canarias',
                  'cantabria',            'castilla_y_leon',
         'castilla_la_mancha',                   'cataluna',
       'comunidad_valenciana',                'extremadura',
                    'galicia',        'comunidad_de_madrid',
           'region_de_murcia', 'comunidad_foral_de_navarra',
                 'pais_vasco',                   'la_rioja',
                      'ceuta',                    'melilla']
Length: 20, dtype: str

Normalizamos resto de columnas (minúsculas, separación con guión bajo ('_'))

In [276]:
# Así se aplica una función a los DATOS de una columna
df_empr_dis["razon"] = df_empr_dis["razon"].apply(trans_normal.normalizar_col)
df_sectores_ipc["nombre_sector"] = df_sectores_ipc["nombre_sector"].apply(trans_normal.normalizar_col)
df_tiempo["nombre_mes"] = df_tiempo["nombre_mes"].apply(trans_normal.normalizar_col)
df_tipo_medida["nombre_medida"] = df_tipo_medida["nombre_medida"].apply(trans_normal.normalizar_col)

In [277]:
df_empr_dis.sample(10)

,id_dis,id_tiempo,razon,numero_sociedades,id_territorio
4686,4687,202010,por_fusion,2,4
10426,10427,201902,otras,39,11
1927,1928,201205,voluntaria,104,10
247,248,202401,voluntaria,142,3
533,534,201807,voluntaria,19,4
2204,2205,202512,voluntaria,52,12
3444,3445,201404,voluntaria,67,17
4221,4222,202211,por_fusion,20,2
1846,1847,201902,voluntaria,107,10
8002,8003,201906,por_fusion,0,19


In [278]:
df_sectores_ipc.sample(10)

,id_sector,nombre_sector
4,5,vivienda_agua_electricidad_gas_y_otros_combust...
12,13,seguros_y_servicios_financieros
1,2,alimentos_y_bebidas_no_alcoholicas
6,7,sanidad
0,1,indice_general
7,8,transporte
5,6,muebles_articulos_del_hogar_y_articulos_para_e...
10,11,ensenanza
13,14,cuidado_personal_proteccion_social_y_bienes_y_...
2,3,bebidas_alcoholicas_y_tabaco


In [279]:
df_tiempo.sample(10)

,id_tiempo,anio,mes,nombre_mes
183,201102,2011,2,febrero
93,201808,2018,8,agosto
156,201305,2013,5,mayo
82,201907,2019,7,julio
127,201510,2015,10,octubre
13,202504,2025,4,abril
154,201307,2013,7,julio
34,202307,2023,7,julio
151,201310,2013,10,octubre
174,201111,2011,11,noviembre


In [280]:
df_tipo_medida.sample(4)

,id_medida,nombre_medida
1,2,variacion_mensual
2,3,variacion_anual
0,1,indice
3,4,variacion_en_lo_que_va_de_ano


Eliminar en empresas_contituidas las tipo mercantiles, son sumatorias del resto de tipo y nos duplican los datos

In [281]:
# Eliminamos las filas que contienen "Mercantiles"
df_empr_const = df_empr_const[df_empr_const['tipo'] != 'Mercantiles']

In [282]:
df_empr_const['tipo'].unique()

<StringArray>
[                   'Sociedades anónimas',
 'Sociedades de responsabilidad limitada',
       'S. Comanditarias y S. Colectivas']
Length: 3, dtype: str

In [283]:
df_empr_const.shape

(12540, 6)

Cambiamos el nombre de las sociedades por sus acrónimos

In [284]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
dicc_siglas = {
    'Sociedades de responsabilidad limitada': 'S.L.',
    'Sociedades anónimas': 'S.A.',
    'S. Comanditarias y S. Colectivas': 'S.Com./S.C.'
}

In [285]:
# Aplicamos el cambio a la columna 'tipo'
df_empr_const['tipo'] = df_empr_const['tipo'].replace(dicc_siglas)

In [286]:
df_empr_const['tipo'].unique()

<StringArray>
['S.A.', 'S.L.', 'S.Com./S.C.']
Length: 3, dtype: str

Guardamos los csv's procesados

In [287]:
df_empr_const.to_csv('../files/data_processed/empresas_constituidas.csv', index=False)
df_empr_dis.to_csv('../files/data_processed/empresas_disueltas.csv', index=False)
df_ipc.to_csv('../files/data_processed/ipc.csv', index=False)
df_sectores_ipc.to_csv('../files/data_processed/sectores_ipc.csv', index=False)
df_territorio.to_csv('../files/data_processed/territorio.csv', index=False)
df_tiempo.to_csv('../files/data_processed/tiempo.csv', index=False)
df_tipo_medida.to_csv('../files/data_processed/tipo_medida.csv', index=False)